# 🧠 EduMIND — LoRA Fine-Tuning Embedding Model for Bilingual RAG

> **Dataset**: `corpus.jsonl` — Generated multi-task instruction tuning dataset
> **Framework**: `sentence-transformers` + PEFT LoRA (via `GISTEmbedLoss` / `MultipleNegativesRankingLoss`)
> **Target**: Fine-tune a bilingual embedding model for cross-lingual semantic search in EduMIND RAG pipeline
> **Hardware**: Kaggle 2× NVIDIA T4

---

## 🗺️ Notebook Structure

1. **Environment Setup** — Install sentence-transformers, verify GPUs
2. **Dataset Loading** — Load local or Kaggle input `corpus.jsonl`
3. **Base Model** — Load a multilingual embedding model
4. **LoRA on Embedding** — Apply PEFT adapters to encoder
5. **Training** — Fine-tune with `MatryoshkaLoss` + `MultipleNegativesRankingLoss`
6. **Evaluation** — Semantic similarity benchmarks (cross-lingual)
7. **Save & Export** — Push to HuggingFace Hub
8. **EduMIND Integration** — How to update `.env` to use the new embedder

## 1️⃣ Environment Setup


In [ ]:
import subprocess, sys, os
import torch

# GPU check
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu_info.stdout)

NUM_GPUS = torch.cuda.device_count()
print(f"✅ Found {NUM_GPUS} GPU(s)")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
!pip install --quiet \
    "sentence-transformers>=3.0.0" \
    datasets \
    peft \
    accelerate \
    evaluate \
    huggingface_hub \
    mteb

print("✅ Dependencies installed")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

# ── Base embedding model ─────────────────────────────────────────────────────
# Recommended multilingual models for Vietnamese support:
#   - "BAAI/bge-m3"                        (best quality, supports 100+ languages)
#   - "intfloat/multilingual-e5-large"      (good baseline, 560M params)
#   - "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  (faster/lighter)
#   - "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"  (Vietnamese-specific)
BASE_MODEL_ID   = "BAAI/bge-m3"

# ── Output paths ────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/edumind-vietmix-embedding"
HF_REPO_ID      = "Khang3004/edumind-vietmix-bge-m3-lora"  # ← change this

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE      = 32       # larger batches = more negatives for contrastive loss
EPOCHS          = 5
LEARNING_RATE   = 2e-5
WARMUP_RATIO    = 0.1
EVAL_SPLIT      = 0.1
SEED            = 42

# ── LoRA config ──────────────────────────────────────────────────────────────
LORA_R          = 8        # lower rank for encoder models (they're smaller)
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
# For encoder models: target self-attention layers
LORA_TARGETS    = ["query", "key", "value"]  # standard BERT-style naming

# ── Matryoshka embedding dimensions ─────────────────────────────────────────
# Train with multiple output dimensions simultaneously for flexibility
MATRYOSHKA_DIMS = [1024, 512, 256, 128, 64]

# ── HF token ────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except ImportError:
    # Fallback to local environment variables if not running on Kaggle
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📋 Config:")
print(f"   Base model      : {BASE_MODEL_ID}")
print(f"   LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Batch size      : {BATCH_SIZE}")
print(f"   Epochs          : {EPOCHS}")
print(f"   Matryoshka dims : {MATRYOSHKA_DIMS}")

## 2️⃣ Dataset Loading & Contrastive Pair Construction


In [ ]:
# ── Load Multi-Task Corpus ───────────────────────────────────────────────────
import os
from datasets import load_dataset
import pandas as pd

# Determine dataset path (local workspace vs Kaggle Input)
dataset_path = "data/processed/corpus.jsonl"
if not os.path.exists(dataset_path):
    # Search in Kaggle inputs
    kaggle_input_dir = "/kaggle/input"
    found = False
    if os.path.exists(kaggle_input_dir):
        for root, dirs, files in os.walk(kaggle_input_dir):
            for file in files:
                if file == "corpus.jsonl":
                    dataset_path = os.path.join(root, file)
                    found = True
                    break
            if found:
                break

print(f"📥 Loading multitask dataset from: {dataset_path}")
raw_dataset = load_dataset("json", data_files=dataset_path, split="train")
print("\n📊 Dataset info:")
print(raw_dataset)

# Preview first 5 examples
df_preview = raw_dataset.to_pandas().head(5)
print(f"\n📋 First 5 examples:")
pd.set_option('display.max_colwidth', 120)
print(df_preview)

In [ ]:
# ── Explore Task Distribution ────────────────────────────────────────────────
df = raw_dataset.to_pandas()
print(f"📊 Total examples: {len(df):,}")
print("\n🔀 Distribution by task:")
print(df["task"].value_counts())

In [ ]:
# ── Build Contrastive Training Pairs ─────────────────────────────────────────
# For embedding models, we use:
#   POSITIVE PAIRS: (input, output) from semantic_alignment tasks
#   This aligns the Vietnamese code-mixed query space with the English document space.
#   NEGATIVE PAIRS: automatically mined in-batch by MultipleNegativesRankingLoss.
from datasets import Dataset

def build_embedding_dataset(raw_ds) -> Dataset:
    """Extract semantic alignment pairs for contrastive learning."""
    pairs = []
    for row in raw_ds:
        # We only use semantic_alignment records for cross-lingual alignment
        if row.get("task") != "semantic_alignment":
            continue
            
        anchor = str(row.get("input") or "").strip()
        positive = str(row.get("output") or "").strip()
        
        if len(anchor) < 5 or len(positive) < 5:
            continue
            
        # Direction 1: VI_mix -> EN
        pairs.append({"anchor": anchor, "positive": positive})
        # Direction 2: EN -> VI_mix
        pairs.append({"anchor": positive, "positive": anchor})
        
    return Dataset.from_list(pairs)

print("⚙️  Building contrastive training dataset...")
full_emb_dataset = build_embedding_dataset(raw_dataset)
print(f"✅ Total contrastive pairs: {len(full_emb_dataset):,}")

# Train/eval split
split = full_emb_dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)
train_emb = split["train"]
eval_emb  = split["test"]

print(f"   Train size: {len(train_emb):,}")
print(f"   Eval size : {len(eval_emb):,}")

# Preview
print("\n📋 Sample pairs:")
for ex in full_emb_dataset.select(range(3)):
    print(f"  Anchor  : {ex['anchor'][:80]}")
    print(f"  Positive: {ex['positive'][:80]}")
    print()

## 3️⃣ Load Base Embedding Model


In [ ]:
from sentence_transformers import SentenceTransformer

print(f"📥 Loading base embedding model: {BASE_MODEL_ID}")
base_model = SentenceTransformer(
    BASE_MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN if HF_TOKEN else None,
)

print(f"\n✅ Model loaded!")
print(f"   Max seq length   : {base_model.max_seq_length}")
print(f"   Embedding dim    : {base_model.get_sentence_embedding_dimension()}")

# VRAM
for i in range(NUM_GPUS):
    used = torch.cuda.memory_allocated(i) / 1e9
    total_vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.2f} / {total_vram:.1f} GB")

# Quick embedding test
test_vi = "Machine learning là một lĩnh vực quan trọng trong AI."
test_en = "Machine learning is an important field in AI."
emb_vi = base_model.encode(test_vi, convert_to_tensor=True)
emb_en = base_model.encode(test_en, convert_to_tensor=True)
from sentence_transformers import util
sim = util.cos_sim(emb_vi, emb_en).item()
print(f"\n🔍 Baseline cross-lingual similarity (VI↔EN): {sim:.4f}")
print("   (target: >0.85 after fine-tuning)")

## 4️⃣ Apply LoRA Adapters to Encoder


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ── Apply LoRA to the underlying transformer encoder ─────────────────────────
# Note: sentence-transformers wraps a HuggingFace model; we apply LoRA to it directly

encoder = base_model[0].auto_model  # the underlying HuggingFace model

# Inspect architecture to find correct target modules
print("🔍 Encoder architecture (first few layers):")
for name, module in list(encoder.named_modules())[:30]:
    if any(t in name for t in ["query", "key", "value", "dense", "attention"]):
        print(f"  {name}: {type(module).__name__}")

print(f"\n⚙️  Applying LoRA with target_modules={LORA_TARGETS}")

In [ ]:
# ── LoRA config for encoder models ────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # embedding task
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    use_rslora=True,       # rank-stabilized LoRA
    init_lora_weights="gaussian",
)

# Apply LoRA
lora_encoder = get_peft_model(encoder, lora_config)

# Update the sentence-transformers model to use the LoRA encoder
base_model[0].auto_model = lora_encoder

# Parameter stats
total_params     = sum(p.numel() for p in lora_encoder.parameters())
trainable_params = sum(p.numel() for p in lora_encoder.parameters() if p.requires_grad)
print(f"📊 LoRA Parameter summary:")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Frozen params    : {total_params - trainable_params:,}")

## 5️⃣ Training with Matryoshka + MNRL Loss


In [ ]:
from sentence_transformers import losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# ── Loss function: Matryoshka + MultipleNegativesRanking ─────────────────────
# MultipleNegativesRankingLoss (MNRL):
#   - In each batch, each (anchor, positive) pair treats ALL other positives as negatives
#   - Very effective for contrastive learning with bilingual pairs
#   - Requires LARGE batch sizes (32+ recommended)

# MatryoshkaLoss:
#   - Trains multiple embedding dimensions simultaneously
#   - Allows runtime dimension selection (trade quality vs speed)
#   - Essential for flexible deployment in EduMIND RAG

inner_loss = losses.MultipleNegativesRankingLoss(
    model=base_model,
    scale=20.0,         # temperature scaling
    similarity_fct=util.cos_sim,
)

train_loss = losses.MatryoshkaLoss(
    model=base_model,
    loss=inner_loss,
    matryoshka_dims=MATRYOSHKA_DIMS,
)

print("✅ Loss functions configured:")
print(f"   Inner: MultipleNegativesRankingLoss (scale=20.0)")
print(f"   Outer: MatryoshkaLoss (dims={MATRYOSHKA_DIMS})")

In [ ]:
# ── Build evaluator using eval set ────────────────────────────────────────────
# We evaluate cosine similarity between anchor and positive pairs

eval_anchors   = eval_emb["anchor"]
eval_positives = eval_emb["positive"]

# All pairs have score 1.0 (they are semantically equivalent translations)
eval_scores = [1.0] * len(eval_anchors)

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=eval_anchors,
    sentences2=eval_positives,
    scores=eval_scores,
    name="vietmix_eval",
    write_csv=True,
)

# Run evaluator on base model (before fine-tuning)
baseline_score = evaluator(base_model)
print(f"📊 Baseline Pearson correlation (before fine-tuning): {baseline_score:.4f}")

In [ ]:
# ── Training arguments ────────────────────────────────────────────────────────
training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Batch & epochs
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    
    # Learning rate
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    
    # Precision (T4 supports float16 but not bfloat16)
    fp16=True,
    bf16=False,
    
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="vietmix_eval_pearson_cosine",
    greater_is_better=True,
    
    # Logging
    logging_steps=100,
    report_to="none",
    
    seed=SEED,
    optim="adamw_torch",
    max_grad_norm=1.0,
    
    # Multi-GPU: DataParallel is used automatically
    dataloader_num_workers=2,
    
    push_to_hub=False,
)

eff_batch = BATCH_SIZE * NUM_GPUS
steps_per_epoch = len(train_emb) // eff_batch
print(f"✅ Training args configured")
print(f"   Effective batch size : {eff_batch}")
print(f"   Steps per epoch      : {steps_per_epoch:,}")
print(f"   Total steps          : {steps_per_epoch * EPOCHS:,}")

In [ ]:
import time

# ── Build Trainer ──────────────────────────────────────────────────────────────
trainer = SentenceTransformerTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_emb,
    eval_dataset=eval_emb,
    loss=train_loss,
    evaluator=evaluator,
)

# ── Train ────────────────────────────────────────────────────────────────────
print("🚀 Starting Embedding LoRA fine-tuning...")
print(f"   Estimated time: ~{steps_per_epoch * EPOCHS // 60 + 1} minutes on 2×T4")
print("─" * 60)

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"\n✅ Training complete! ({elapsed/60:.1f} minutes)")

## 6️⃣ Evaluation — Cross-Lingual Semantic Similarity


In [ ]:
# ── Final evaluation ─────────────────────────────────────────────────────────
print("📊 Running final evaluation...")
final_score = evaluator(base_model)
print(f"\n📊 Final Pearson correlation (after fine-tuning): {final_score:.4f}")
print(f"📈 Improvement: +{(final_score - baseline_score):.4f} over baseline")

In [ ]:
# ── Qualitative evaluation: cross-lingual retrieval ──────────────────────────
import torch.nn.functional as F

TEST_QUERIES_VI = [
    "Machine learning được ứng dụng rộng rãi trong nhiều lĩnh vực.",
    "Neural network có khả năng học được các đặc trưng phức tạp.",
    "Transformer architecture đã thay đổi cách chúng ta xử lý ngôn ngữ.",
    "Fine-tuning giúp cải thiện performance của model trên domain cụ thể.",
    "Embedding là cách biểu diễn từ ngữ dưới dạng vector số.",
]

TEST_DOCS_EN = [
    "Machine learning is widely applied in many different fields.",
    "Neural networks can learn complex feature representations.",
    "The Transformer architecture revolutionized natural language processing.",
    "Fine-tuning improves model performance on specific domains.",
    "Embeddings represent words and phrases as numerical vectors.",
    # Distractors (semantically different)
    "The weather today is sunny and warm.",
    "She went to the grocery store to buy vegetables.",
    "The economic forecast for next quarter remains uncertain.",
]

# Encode queries and docs
query_embs = base_model.encode(TEST_QUERIES_VI, convert_to_tensor=True, normalize_embeddings=True)
doc_embs   = base_model.encode(TEST_DOCS_EN,   convert_to_tensor=True, normalize_embeddings=True)

# Similarity matrix
sim_matrix = util.cos_sim(query_embs, doc_embs)

print("🔍 Cross-Lingual Retrieval Results (VI query → EN documents):")
print("─" * 70)
correct = 0
for i, query in enumerate(TEST_QUERIES_VI):
    sims = sim_matrix[i].cpu().numpy()
    top_idx = sims.argmax()
    is_correct = top_idx == i  # first 5 docs match first 5 queries
    if is_correct:
        correct += 1
    
    print(f"\n[{'✅' if is_correct else '❌'}] Query: {query[:60]}")
    print(f"     Top-1 match: {TEST_DOCS_EN[top_idx][:60]} (sim={sims[top_idx]:.3f})")

print(f"\n📊 Top-1 Accuracy: {correct}/{len(TEST_QUERIES_VI)} ({100*correct/len(TEST_QUERIES_VI):.0f}%)")

In [ ]:
# ── Matryoshka dimension analysis ────────────────────────────────────────────
# Evaluate quality at each Matryoshka dimension
print("📊 Matryoshka Dimension Analysis:")
print("   Testing cross-lingual similarity at different dimensions")
print("─" * 50)

for dim in MATRYOSHKA_DIMS:
    # Truncate embeddings to this dimension
    q_trunc = F.normalize(query_embs[:, :dim], dim=-1)
    d_trunc = F.normalize(doc_embs[:, :dim],   dim=-1)
    sim_trunc = util.cos_sim(q_trunc, d_trunc)
    
    # Top-1 accuracy
    correct_dim = sum(
        1 for i in range(len(TEST_QUERIES_VI))
        if sim_trunc[i].argmax().item() == i
    )
    avg_sim = sim_trunc.diagonal()[:len(TEST_QUERIES_VI)].mean().item()
    
    print(f"   Dim {dim:4d}: Top-1={correct_dim}/{len(TEST_QUERIES_VI)} | Avg paired sim={avg_sim:.4f}")

## 7️⃣ Save & Export


In [ ]:
# ── Save the fine-tuned model ─────────────────────────────────────────────────
SAVE_PATH = f"{OUTPUT_DIR}/final_model"

# Option A: Save merged model (LoRA merged into base — standalone inference)
print("💾 Merging LoRA and saving full model...")

# Merge LoRA weights into the encoder
from peft import PeftModel
merged_encoder = lora_encoder.merge_and_unload()
base_model[0].auto_model = merged_encoder

# Save the sentence-transformers model
base_model.save_pretrained(SAVE_PATH)
print(f"✅ Saved to: {SAVE_PATH}")

# List files
saved_files = os.listdir(SAVE_PATH)
total_size = sum(os.path.getsize(f"{SAVE_PATH}/{f}") for f in saved_files if os.path.isfile(f"{SAVE_PATH}/{f}"))
print(f"   Files: {len(saved_files)} | Size: {total_size / 1e6:.1f} MB")

In [ ]:
# ── Push to HuggingFace Hub ───────────────────────────────────────────────────
if HF_TOKEN and HF_REPO_ID != "YOUR_HF_USERNAME/edumind-vietmix-bge-m3-lora":
    from huggingface_hub import login
    login(token=HF_TOKEN)
    
    print(f"📤 Pushing to Hub: {HF_REPO_ID}")
    base_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f"✅ Uploaded: https://huggingface.co/{HF_REPO_ID}")
else:
    print("⚠️  Skipping Hub push — set HF_TOKEN and update HF_REPO_ID")

## 8️⃣ EduMIND Integration


In [ ]:
# ── How to update EduMIND to use the fine-tuned embedding model ───────────────

ENV_CONFIG = """
# ── EduMIND Embedding Configuration ────────────────────────────────────────
# Use fine-tuned bilingual embedding model for better RAG accuracy

# Option 1: Load from HuggingFace Hub (after pushing)
EDUMIND_EMBEDDING_MODEL=YOUR_HF_USERNAME/edumind-vietmix-bge-m3-lora

# Option 2: Load from local path (Kaggle output)
# EDUMIND_EMBEDDING_MODEL=/path/to/edumind-vietmix-embedding/final_model

# Option 3: Fallback to original model (no fine-tuning)
# EDUMIND_EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2

# Matryoshka dimension (optional — use smaller for speed)
# EDUMIND_EMBEDDING_DIMENSION=256  # options: 1024, 512, 256, 128, 64
"""

print("📋 EduMIND .env configuration:")
print(ENV_CONFIG)

print("\n📝 After updating .env:")
print("   1. Restart EduMIND containers: docker compose restart")
print("   2. Re-index existing documents in Qdrant (rebuild vector index)")
print("   3. Test with a bilingual query to verify improvement")
print("\n⚠️  Note: Changing embedding model requires re-indexing ALL documents")
print("   because the vector space changes between models!")

In [ ]:
# ── Final demo: Load saved model and run inference ────────────────────────────
from sentence_transformers import SentenceTransformer

# Load the saved fine-tuned model
finetuned_model = SentenceTransformer(SAVE_PATH, trust_remote_code=True)

# Simulate EduMIND RAG query
QUERY_VI = "Giải thích về attention mechanism trong transformer."
LECTURE_CHUNKS_EN = [
    "The attention mechanism allows the model to focus on relevant parts of the input sequence.",
    "Transformers use self-attention to compute representations of input sequences.",
    "The economic impact of renewable energy is significant in developing countries.",  # distractor
    "Multi-head attention computes attention in multiple subspaces simultaneously.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",  # distractor
]

q_emb = finetuned_model.encode(QUERY_VI, normalize_embeddings=True)
d_embs = finetuned_model.encode(LECTURE_CHUNKS_EN, normalize_embeddings=True)
sims = (q_emb @ d_embs.T)

print("🔍 EduMIND RAG Simulation:")
print(f"   Query (VI): {QUERY_VI}")
print(f"\n   Retrieved chunks (ranked by similarity):")
ranked = sorted(zip(sims, LECTURE_CHUNKS_EN), reverse=True)
for rank, (sim, chunk) in enumerate(ranked, 1):
    print(f"   [{rank}] sim={sim:.4f} | {chunk[:80]}")